In [1]:
import gymnasium as gym
import minigrid
import os

env = gym.make("MiniGrid-Empty-5x5-v0")

obs, info = env.reset()

print(f'observation_space: {env.observation_space}')
print(f'action_space: {env.action_space}')

print(f'obs keys: {obs.keys()}')
print(f'imag shape:', {obs['image'].shape})
print(f"mission: {obs['mission']}")
print(f"direction: {obs['direction']}")

observation_space: Dict('direction': Discrete(4), 'image': Box(0, 255, (7, 7, 3), uint8), 'mission': MissionSpace(<function EmptyEnv._gen_mission at 0x1308ce290>, None))
action_space: Discrete(7)
obs keys: dict_keys(['image', 'direction', 'mission'])
imag shape: {(7, 7, 3)}
mission: get to the green goal square
direction: 0


/Users/k.choi/opt/miniconda3/envs/pytorch_csiro/lib/python3.10/site-packages/pygame/pkgdata.py:25: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import resource_stream, resource_exists


In [2]:
from minigrid.wrappers import ImgObsWrapper

env = gym.make("MiniGrid-Empty-5x5-v0")
obs, _ = env.reset()
print(obs.keys())
env = ImgObsWrapper(env)
obs, _ = env.reset()
print(obs.shape)



dict_keys(['image', 'direction', 'mission'])
(7, 7, 3)


In [3]:
import torch 
import torch.nn as nn
import gymnasium as gym
from stable_baselines3.common.torch_layers import BaseFeaturesExtractor

class MiniGridFeaturesExtractor(BaseFeaturesExtractor):
    def __init__(self, obs_space: gym.Space, features_dim: int = 512, normalized_image: bool = False) -> None:
        super().__init__(obs_space, features_dim)

        n_input_channels = obs_space.shape[0]

        self.cnn = nn.Sequential(
            nn.Conv2d(n_input_channels,16, (2,2)),
            nn.ReLU(),
            nn.Conv2d(16,32, (2,2)),
            nn.ReLU(),
            nn.Conv2d(32,64, (2,2)),
            nn.ReLU(),
            nn.Flatten(),
        )

        with torch.no_grad():
            n_flatten = self.cnn(torch.as_tensor(obs_space.sample()[None]).float()).shape[1]
        self.linear = nn.Sequential(nn.Linear(n_flatten, features_dim), nn.ReLU())
    
    def forward(self, observations: torch.Tensor) -> torch.Tensor:
        return self.linear(self.cnn(observations))


In [4]:
from pyexpat import model
from stable_baselines3.common.callbacks import EvalCallback
from stable_baselines3.common.monitor import Monitor
from stable_baselines3.common.env_util import make_vec_env



env_names_list = [
    "MiniGrid-Empty-5x5-v0",
    "MiniGrid-Empty-8x8-v0",
    "MiniGrid-Empty-16x16-v0", 
    "MiniGrid-DoorKey-5x5-v0",
    "MiniGrid-DoorKey-8x8-v0",
    "MiniGrid-DoorKey-16x16-v0"
]

env_name = env_names_list[4]; print(env_name)

results = {}
log_dir = os.path.join(os.getcwd(), "logs/ppo_minigrid")
model_dir = os.path.join(os.getcwd(), "models/ppo_minigrid")
video_dir = os.path.join(os.getcwd(), "videos/ppo_minigrid")

if not os.path.exists(log_dir):
    os.makedirs(log_dir, exist_ok=True)
if not os.path.exists(model_dir):
    os.makedirs(model_dir, exist_ok=True)
if not os.path.exists(video_dir):
    os.makedirs(video_dir, exist_ok=True)

 
def get_train_env(env_name):
    env = gym.make(f"{env_name}", render_mode="rgb_array")
    env = ImgObsWrapper(env)
    env = Monitor(env)
    return env

def get_eval_env(env_name):
    env = gym.make(f"{env_name}", render_mode="rgb_array")
    env = ImgObsWrapper(env)
    env = Monitor(env)
    return env

train_env = get_train_env(env_name)
# eval_env = get_eval_env(env_name)
# train_env = make_vec_env(lambda: get_train_env(env_name), n_envs=3)
eval_env = get_eval_env(env_name)


eval_callback = EvalCallback(
    eval_env,
    best_model_save_path=model_dir,
    log_path=log_dir,
    eval_freq=10000,
    deterministic=True,
    render=False,
)

MiniGrid-DoorKey-8x8-v0


In [5]:
import gymnasium as gym
from stable_baselines3 import PPO

policy_kwargs = dict(
    features_extractor_class=MiniGridFeaturesExtractor,
    features_extractor_kwargs=dict(features_dim=128),
)

env = gym.make(f"{env_name}", render_mode="rgb_array")
env = ImgObsWrapper(env)

model = PPO(
    policy="CnnPolicy",
    env=train_env,
    policy_kwargs=policy_kwargs,
    ent_coef = 0.01,
    # learning_rate=1e-3, 
    verbose=1,
)

model.learn(total_timesteps=1e5, callback=eval_callback)


Using cpu device
Wrapping the env in a DummyVecEnv.
Wrapping the env in a VecTransposeImage.


/Users/k.choi/opt/miniconda3/envs/pytorch_csiro/lib/python3.10/site-packages/stable_baselines3/common/callbacks.py:418: UserWarning: Training and eval env are not of the same type<stable_baselines3.common.vec_env.vec_transpose.VecTransposeImage object at 0x120636470> != <stable_baselines3.common.vec_env.dummy_vec_env.DummyVecEnv object at 0x15157b0a0>
  warnings.warn("Training and eval env are not of the same type" f"{self.training_env} != {self.eval_env}")


---------------------------------
| rollout/           |          |
|    ep_len_mean     | 607      |
|    ep_rew_mean     | 0.0802   |
| time/              |          |
|    fps             | 2267     |
|    iterations      | 1        |
|    time_elapsed    | 0        |
|    total_timesteps | 2048     |
---------------------------------
-----------------------------------------
| rollout/                |             |
|    ep_len_mean          | 623         |
|    ep_rew_mean          | 0.0401      |
| time/                   |             |
|    fps                  | 983         |
|    iterations           | 2           |
|    time_elapsed         | 4           |
|    total_timesteps      | 4096        |
| train/                  |             |
|    approx_kl            | 0.005558077 |
|    clip_fraction        | 0.0268      |
|    clip_range           | 0.2         |
|    entropy_loss         | -1.94       |
|    explained_variance   | -0.0727     |
|    learning_rate        | 0.

In [6]:
from stable_baselines3.common.evaluation import evaluate_policy
from stable_baselines3.common.monitor import Monitor
import numpy as np


episode_rewards, episode_lengths = evaluate_policy(
    model,
    eval_env,
    n_eval_episodes=20,
    deterministic=True,
    return_episode_rewards=True
    )

print(f"Mean reward: { np.mean(episode_rewards):.3f} +/- {np.std(episode_rewards)  :.3f}")
print(f"Mean episode length: {np.mean(episode_lengths):.3f} +/- {np.std(episode_lengths):.3f}")

successes = [r > 0 for r in episode_rewards]
success_rate = np.mean(successes)

print(f"Success rate: {100 * success_rate:.1f}%")


Mean reward: 0.000 +/- 0.000
Mean episode length: 640.000 +/- 0.000
Success rate: 0.0%


In [ ]:
random_rewards = []
random_lengths = []

baseline_env = gym.make(f"{env_name}")
baseline_env = ImgObsWrapper(baseline_env)
baseline_env = Monitor(baseline_env)

for _ in range(50):
    obs, info = baseline_env.reset()
    done = False
    total_reward = 0
    steps = 0

    while not done:
        action = baseline_env.action_space.sample()
        obs, reward, terminated, truncated, info = baseline_env.step(action)
        done = terminated or truncated
        total_reward += reward
        steps += 1

    random_rewards.append(total_reward)
    random_lengths.append(steps)

baseline_env.close()

print("Random mean reward:", np.mean(random_rewards))
print("Random success rate:", 100*np.mean(np.array(random_rewards) > 0))
print("Random mean length:", np.mean(random_lengths))


results[env_name] = {
    "mean_reward": np.mean(episode_rewards),
    "std_reward": np.std(episode_rewards),
    "mean_episode_length": np.mean(episode_lengths),
    "std_episode_length": np.std(episode_lengths),
    "success_rate": 100 * success_rate,
    "random_mean_reward": np.mean(random_rewards),
    "random_success_rate": 100*np.mean(np.array(random_rewards) > 0),
    "random_mean_length": np.mean(random_lengths)
}

In [ ]:
import os
from minigrid.wrappers import ImgObsWrapper
from gymnasium.wrappers import RecordVideo

main_dir = os.getcwd()
model_dir = os.path.join(main_dir, "models/ppo_minigrid")
video_dir = os.path.join(main_dir, "videos/ppo_minigrid")

os.makedirs(model_dir, exist_ok=True)
os.makedirs(video_dir, exist_ok=True)

video_env = gym.make(f"{env_name}", render_mode="rgb_array")
video_env = ImgObsWrapper(video_env)

video_env = RecordVideo(
    video_env, 
    video_dir, 
    episode_trigger=lambda ep: ep < 3,   # record first 5 episodes
    name_prefix=f"{env_name}_ppo")

for ep in range(3):
    obs, info = video_env.reset()
    done = False
    total_reward = 0
    while not done:
        action, _ = model.predict(obs, deterministic=True)
        obs, reward, terminated, truncated, info = video_env.step(action)
        done = terminated or truncated
        total_reward += reward
video_env.close()
print(f"Episode {ep} finished with reward {total_reward}"
)
        